# ColonyNet ? Unified Training (MiT-B3)

???????? ?? ???????????? ???????? `data/unified` ? ??????? ?????????????.
? ???????? ???????? ??????? ? ??????????? ETA.

**Note:** MiT-B3 is heavier; default `batch_size=4` to fit RTX 3060.


In [ ]:
import os, time, math, yaml
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

from colonyseg.utils import set_seed, ensure_dir
from colonyseg.data.datasets import ImageInstancesDataset, split_ids
from colonyseg.data.transforms import build_train_tf, build_val_tf
from colonyseg.models.colonymet import ColonyNet, set_backbone_trainable
from colonyseg.losses import loss_total
from colonyseg.post.watershed import postprocess_watershed
from colonyseg.metrics.instance_metrics import instance_scores


In [ ]:
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
def load_yaml(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

cfg_path = 'configs/train_trainable_pool_mit_b3_edges_safe.yaml'
cfg = load_yaml(cfg_path)

target_cfg = cfg.get('targets', None)
cfg


In [ ]:
set_seed(int(cfg.get('seed', 42)))

run_dir = os.path.join('runs', cfg['run_name'])
ensure_dir(run_dir)

img_size = int(cfg['data']['img_size'])
out_stride = int(cfg['data']['out_stride'])
train_tf = build_train_tf(img_size)
val_tf = build_val_tf(img_size)

all_imgs = sorted([p for p in os.listdir(cfg['data']['train_images']) if p.lower().endswith(('.png','.jpg','.jpeg','.tif','.tiff','.bmp'))])
all_ids = [os.path.splitext(p)[0] for p in all_imgs]
train_ids, val_ids = split_ids(all_ids, float(cfg['data']['val_split']), seed=int(cfg.get('seed', 42)))

train_ds = ImageInstancesDataset(
    cfg['data']['train_images'], cfg['data']['train_instances'],
    transform=train_tf, img_size=img_size, out_stride=out_stride, ids=train_ids, target_cfg=target_cfg
)
val_ds = ImageInstancesDataset(
    cfg['data']['val_images'], cfg['data']['val_instances'],
    transform=val_tf, img_size=img_size, out_stride=out_stride, ids=val_ids, target_cfg=target_cfg
)

train_loader = DataLoader(train_ds, batch_size=int(cfg['train']['batch_size']), shuffle=True,
                          num_workers=int(cfg['train']['num_workers']), pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=max(1, int(cfg['train']['num_workers'])//2), pin_memory=True)

print('train:', len(train_ds), 'val:', len(val_ds))


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ColonyNet(backbone_id=cfg['model']['backbone_id'], fpn_dim=int(cfg['model']['fpn_dim'])).to(device)

lr_head = float(cfg['train']['lr_head'])
lr_backbone = float(cfg['train']['lr_backbone'])
wd = float(cfg['train']['weight_decay'])

head_params = []
backbone_params = []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if n.startswith('backbone.'):
        backbone_params.append(p)
    else:
        head_params.append(p)

optim = torch.optim.AdamW(
    [
        {'params': head_params, 'lr': lr_head},
        {'params': backbone_params, 'lr': lr_backbone},
    ],
    weight_decay=wd
)

scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg['train']['amp']))

print('device:', device)


In [ ]:
epochs = int(cfg['train']['epochs'])
freeze_epochs = int(cfg['train']['freeze_backbone_epochs'])
loss_weights = cfg['train'].get('loss_weights', None)
boundary_dice = float(cfg['train'].get('boundary_dice', 0.0))
best_f1 = -1.0

history = {
    'train_loss': [],
    'val_f1': [],
    'val_merge': [],
    'val_split': [],
    'val_count_err': [],
    'epoch_time': [],
}

plot_handle = None

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    # Freeze/unfreeze schedule
    if epoch <= freeze_epochs:
        set_backbone_trainable(model, False)
    else:
        set_backbone_trainable(model, True)

    model.train()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [train]')
    loss_sum = 0.0
    n_batches = 0

    for batch in pbar:
        x = batch['image'].to(device, non_blocking=True)
        y_sem = batch['y_sem'].to(device, non_blocking=True)
        y_center = batch['y_center'].to(device, non_blocking=True)
        y_boundary = batch['y_boundary'].to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=bool(cfg['train']['amp'])):
            pred = model(x)
            loss, parts = loss_total(pred, y_sem, y_center, y_boundary, weights=loss_weights, boundary_dice=boundary_dice)

        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()

        loss_sum += loss.item()
        n_batches += 1
        pbar.set_postfix(loss='{:.4f}'.format(loss.item()), **{k: '{:.3f}'.format(v) for k, v in parts.items()})

    train_loss = loss_sum / max(1, n_batches)

    # Validation: postprocess + instance metrics
    model.eval()
    metrics = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{epochs} [val]'):
            x = batch['image'].to(device, non_blocking=True)
            gt_inst = batch['instances'].cpu().numpy()[0]
            pred = model(x)

            sem_p = torch.sigmoid(pred['sem']).cpu().numpy()[0,0]
            cen_p = torch.sigmoid(pred['center']).cpu().numpy()[0,0]
            bnd_p = torch.sigmoid(pred['boundary']).cpu().numpy()[0,0]

            pr_labels = postprocess_watershed(
                sem_p, cen_p, bnd_p,
                t_sem=float(cfg['post']['t_sem']),
                t_center=float(cfg['post']['t_center']),
                min_distance=int(cfg['post']['min_distance']),
                lambda_boundary=float(cfg['post']['lambda_boundary']),
                area_min=int(cfg['post']['area_min']),
                area_max=int(cfg['post']['area_max']),
            )

            H, W = gt_inst.shape
            out_h, out_w = sem_p.shape
            import cv2
            gt_small = cv2.resize(gt_inst.astype(np.int32), (out_w, out_h), interpolation=cv2.INTER_NEAREST)

            m = instance_scores(gt_small, pr_labels, iou_thr=float(cfg['train']['iou_thr']))
            metrics.append(m)

    mean_f1 = float(np.mean([m['f1'] for m in metrics])) if metrics else 0.0
    mean_mer = float(np.mean([m['merge'] for m in metrics])) if metrics else 0.0
    mean_spl = float(np.mean([m['split'] for m in metrics])) if metrics else 0.0
    mean_cnt = float(np.mean([m['count_err'] for m in metrics])) if metrics else 0.0

    # Save
    last_path = os.path.join(run_dir, 'last.pt')
    torch.save({'epoch': epoch, 'model': model.state_dict(), 'cfg': cfg}, last_path)
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_path = os.path.join(run_dir, 'best.pt')
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'cfg': cfg}, best_path)

    # Timing + history
    epoch_time = time.time() - epoch_start
    history['train_loss'].append(train_loss)
    history['val_f1'].append(mean_f1)
    history['val_merge'].append(mean_mer)
    history['val_split'].append(mean_spl)
    history['val_count_err'].append(mean_cnt)
    history['epoch_time'].append(epoch_time)

    avg_epoch = float(np.mean(history['epoch_time'][-5:]))
    eta_sec = avg_epoch * (epochs - epoch)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()
    ax[1].plot(history['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()
    if plot_handle is None:
        plot_handle = display(fig, display_id=True)
    else:
        plot_handle.update(fig)
    plt.close(fig)

    print('Epoch {}/{} | train_loss={:.4f} | val_f1={:.4f} | merge={:.3f} | split={:.3f} | count_err={:.3f}'.format(
        epoch, epochs, train_loss, mean_f1, mean_mer, mean_spl, mean_cnt
    ))
    print('Epoch time: {:.1f}s | Avg (last 5): {:.1f}s | ETA: {:.1f} min'.format(
        epoch_time, avg_epoch, eta_sec/60.0
    ))


In [ ]:
# === paths ===
ckpt_path = 'runs/colony_trainable_pool_mit_b3/best.pt'
img_path  = 'IMG_4377.jpg'

# === load model ===
ckpt = torch.load(ckpt_path, map_location='cpu')
cfg = ckpt['cfg']
model = ColonyNet(backbone_id=cfg['model']['backbone_id'],
                  fpn_dim=int(cfg['model']['fpn_dim'])).to(device)
model.load_state_dict(ckpt['model'], strict=True)
model.eval()

# === petri crop helpers (same logic as tool) ===
import cv2, numpy as np, matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries

def detect_petri_circle(img_rgb, min_r_frac=0.35, max_r_frac=0.55, center_tol=0.25):
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (9, 9), 2)

    min_r = int(min(h, w) * min_r_frac)
    max_r = int(min(h, w) * max_r_frac)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min(h, w)//2,
        param1=100, param2=30, minRadius=min_r, maxRadius=max_r
    )
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        cx, cy, r = circles[np.argmax(circles[:,2])]
    else:
        edges = cv2.Canny(gray, 50, 150)
        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        cnt = max(cnts, key=cv2.contourArea)
        (cx_f, cy_f), r_f = cv2.minEnclosingCircle(cnt)
        cx, cy, r = int(cx_f), int(cy_f), int(r_f)

    if r < min_r or r > max_r:
        return None
    cx0, cy0 = w // 2, h // 2
    max_off = center_tol * min(h, w)
    if ((cx - cx0)**2 + (cy - cy0)**2) ** 0.5 > max_off:
        return None
    return cx, cy, r

def crop_petri(img_rgb, pad=0.02, mask_outside=True):
    h, w = img_rgb.shape[:2]
    circ = detect_petri_circle(img_rgb)
    if circ is None:
        return img_rgb, None, "no_circle"
    cx, cy, r = circ
    r = int(r * (1.0 + pad))

    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)

    crop = img_rgb[y1:y2, x1:x2].copy()

    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        crop[~mask] = 0

    return crop, (cx, cy, r, x1, y1, x2, y2), "cropped"

def overlay_boundaries(img, lbl):
    out = img.copy()
    b = find_boundaries(lbl, mode='outer')
    out[b] = (0, 255, 0)
    return out

# === load image + petri crop ===
img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

img_petri, meta, status = crop_petri(img_rgb, pad=0.02, mask_outside=True)

# === resize to model input ===
img_rs = cv2.resize(img_petri, (cfg['data']['img_size'], cfg['data']['img_size']),
                    interpolation=cv2.INTER_AREA)

# === inference ===
x = torch.from_numpy(img_rs).float().permute(2,0,1) / 255.0
x = x.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)
    sem_p = torch.sigmoid(pred['sem']).cpu().numpy()[0,0]
    cen_p = torch.sigmoid(pred['center']).cpu().numpy()[0,0]
    bnd_p = torch.sigmoid(pred['boundary']).cpu().numpy()[0,0]

# Upsample head outputs to input resolution before watershed for smoother boundaries
h_in, w_in = img_rs.shape[:2]
h_out, w_out = sem_p.shape
scale = h_in / float(h_out)

sem_hi = cv2.resize(sem_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)
cen_hi = cv2.resize(cen_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)
bnd_hi = cv2.resize(bnd_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)

min_distance_hi = max(1, int(round(float(cfg['post']['min_distance']) * scale)))
area_min_hi = max(1, int(round(float(cfg['post']['area_min']) * (scale ** 2))))
area_max_hi = int(round(float(cfg['post']['area_max']) * (scale ** 2)))

labels_up = postprocess_watershed(
    sem_hi, cen_hi, bnd_hi,
    t_sem=float(cfg['post']['t_sem']),
    t_center=float(cfg['post']['t_center']),
    min_distance=min_distance_hi,
    lambda_boundary=float(cfg['post']['lambda_boundary']),
    area_min=area_min_hi,
    area_max=area_max_hi,
)

# === visualize ===
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].set_title('Original')
ax[0].imshow(img_rgb); ax[0].axis('off')

ax[1].set_title(f'Petri-cropped ({status})')
ax[1].imshow(img_petri); ax[1].axis('off')

ax[2].set_title('Pred boundaries')
ax[2].imshow(overlay_boundaries(img_rs, labels_up)); ax[2].axis('off')
plt.show()
